# Notebook d'extraction — Groupe D : fraude à l'ouverture de compte

**Source** : Bank Account Fraud Dataset Suite (Feedzai, NeurIPS 2022), fichier `Base.csv`, téléchargé manuellement sur Kaggle puis découpé en 3 parties (`data/dataset_part1-3.csv`) pour respecter la limite de 100 Mo de GitHub.

**Granularité** : 1 ligne = 1 demande d'ouverture de compte.

**Sorties** : `data/processed/base_clean.parquet`, `lineage.json` (à la racine).

**Exécution** : depuis la racine du dépôt, `Kernel > Restart & Run All`. Dépendances : `pandas`, `numpy`, `pyarrow`.

## 1. Configuration

In [1]:
%pip install -q pandas numpy pyarrow

Note: you may need to restart the kernel to use updated packages.


In [2]:
import hashlib, json, platform
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd()
PARTS = sorted((ROOT / "data").glob("dataset_part*.csv"))
OUT_DIR = ROOT / "data" / "processed"
OUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET = "fraud_bool"
TRAIN_MONTHS = [0, 1, 2, 3, 4, 5]          # test = mois 6 et 7

# Métadonnées de la source (Kaggle)
SOURCE = {
    "nom": "Bank Account Fraud Dataset Suite (Feedzai, NeurIPS 2022) - Base.csv",
    "url": "https://www.kaggle.com/datasets/sgpjesus/bank-account-fraud-dataset-neurips2022",
    "documentation": "https://github.com/feedzai/bank-account-fraud",
    "methode_acces": "Téléchargement manuel de Base.csv sur Kaggle, découpé en 3 parties pour GitHub",
    "version": "1",
    "date_recuperation": "2026-10-08",
    "licence": "CC BY-NC-SA 4.0",
    "sha256_base_csv_reference": "7bf10a37ce07e72e14c1b09e5efee3d27261baff4facc7da767b0474dcf9b809",
}
assert PARTS, "Aucun fichier data/dataset_part*.csv : lancer le notebook depuis la racine du dépôt."
print(f"{len(PARTS)} fichiers trouvés :", [p.name for p in PARTS])

3 fichiers trouvés : ['dataset_part1.csv', 'dataset_part2.csv', 'dataset_part3.csv']


## 2. Vérification d'intégrité (SHA-256)
Les 3 parties, recollées, doivent redonner exactement le `Base.csv` d'origine.

In [3]:
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()

def sha256_parts(paths):
    """SHA-256 des parties recollées (en-tête gardé une seule fois)."""
    h = hashlib.sha256()
    for i, p in enumerate(paths):
        with open(p, "rb") as f:
            if i > 0:
                f.readline()
            for block in iter(lambda: f.read(1 << 20), b""):
                h.update(block)
    return h.hexdigest()

parts_sha = {p.name: sha256_file(p) for p in PARTS}
rebuilt_sha = sha256_parts(PARTS)
assert rebuilt_sha == SOURCE["sha256_base_csv_reference"], "Les parties ne redonnent pas Base.csv !"
print("OK : les parties reconstituent Base.csv à l'identique")

OK : les parties reconstituent Base.csv à l'identique


## 3. Chargement

In [4]:
df = pd.concat((pd.read_csv(p) for p in PARTS), ignore_index=True)
print(df.shape)
assert df.shape == (1_000_000, 32), "Dimensions inattendues"
df.head(3)

(1000000, 32)


,fraud_bool,income,name_email_similarity,prev_address_months_count,current_address_months_count,customer_age,days_since_request,intended_balcon_amount,payment_type,zip_count_4w,...,has_other_cards,proposed_credit_limit,foreign_request,source,session_length_in_minutes,device_os,keep_alive_session,device_distinct_emails_8w,device_fraud_count,month
0,0,0.3,0.986506,-1,25,40,0.006735,102.453711,AA,1059,...,0,1500.0,0,INTERNET,16.224843,linux,1,1,0,0
1,0,0.8,0.617426,-1,89,20,0.010095,-0.849551,AD,1658,...,0,1500.0,0,INTERNET,3.363854,other,1,1,0,0
2,0,0.8,0.996707,9,14,40,0.012316,-1.490386,AB,1095,...,0,200.0,0,INTERNET,22.730559,windows,0,1,0,0


## 4. Contrôles qualité sur les données brutes

In [5]:
quality = {
    "nb_lignes": len(df),
    "nb_colonnes": df.shape[1],
    "nb_fraudes": int(df[TARGET].sum()),
    "taux_fraude_pct": round(df[TARGET].mean() * 100, 3),
    "doublons_exacts": int(df.duplicated().sum()),
    "nan_bruts": int(df.isna().sum().sum()),
    "valeurs_moins_un": {c: int((df[c] == -1).sum()) for c in df.columns if (df[c] == -1).any()},
}
assert set(df[TARGET].unique()) == {0, 1}, "Cible invalide"
print(json.dumps(quality, indent=2, ensure_ascii=False))
print("\nFraudes par mois :")
print(df.groupby("month")[TARGET].agg(["size", "sum", "mean"]).round(4))

{
  "nb_lignes": 1000000,
  "nb_colonnes": 32,
  "nb_fraudes": 11029,
  "taux_fraude_pct": 1.103,
  "doublons_exacts": 0,
  "nan_bruts": 0,
  "valeurs_moins_un": {
    "prev_address_months_count": 712920,
    "current_address_months_count": 4254,
    "credit_risk_score": 488,
    "bank_months_count": 253635,
    "session_length_in_minutes": 2015,
    "device_distinct_emails_8w": 359
  }
}

Fraudes par mois :
         size   sum    mean
month                      
0      132440  1500  0.0113
1      127620  1198  0.0094
2      136979  1198  0.0087
3      150936  1392  0.0092
4      127691  1452  0.0114
5      119323  1411  0.0118
6      108168  1450  0.0134
7       96843  1428  0.0147


## 5. Transformations
Chaque étape est enregistrée dans `TRANSFORMATIONS` (puis dans `lineage.json`).
`credit_risk_score` n'est pas recodé : -1 y est une valeur légitime du score, pas un code d'absence.

In [6]:
TRANSFORMATIONS = []

def log_step(nom, description, avant, apres, **extra):
    TRANSFORMATIONS.append({"etape": len(TRANSFORMATIONS) + 1, "nom": nom, "description": description,
                            "lignes_avant": len(avant), "lignes_apres": len(apres),
                            "colonnes_avant": avant.shape[1], "colonnes_apres": apres.shape[1], **extra})

# 5.1 Doublons exacts (pas d'identifiant disponible)
avant = df
df = df.drop_duplicates().reset_index(drop=True)
log_step("doublons", "Suppression des lignes entièrement identiques", avant, df)

# 5.2 Valeurs manquantes codées -1 (ou négatives pour intended_balcon_amount) -> NaN + indicateur
MANQUANTS = ["prev_address_months_count", "current_address_months_count", "bank_months_count",
             "session_length_in_minutes", "device_distinct_emails_8w"]
avant, recodees = df.copy(), {}
for c in MANQUANTS + ["intended_balcon_amount"]:
    masque = (df[c] < 0) if c == "intended_balcon_amount" else (df[c] == -1)
    recodees[c] = int(masque.sum())
    df[c + "_manquant"] = masque.astype("int8")
    df.loc[masque, c] = np.nan
log_step("recodage_manquants", "-1 (négatifs pour intended_balcon_amount) -> NaN, colonne <col>_manquant ajoutée",
         avant, df, valeurs_recodees=recodees)

# 5.3 Colonnes constantes
constantes = [c for c in df.columns if c != TARGET and df[c].nunique(dropna=False) <= 1]
avant = df
df = df.drop(columns=constantes)
log_step("colonnes_constantes", f"Suppression des colonnes constantes : {constantes}", avant, df,
         colonnes_supprimees=constantes)

# 5.4 Typage
NOMINALES = ["payment_type", "employment_status", "housing_status", "source", "device_os"]
BINAIRES = [TARGET, "email_is_free", "phone_home_valid", "phone_mobile_valid", "has_other_cards",
            "foreign_request", "keep_alive_session"]
avant = df
df[NOMINALES] = df[NOMINALES].astype("category")
df[BINAIRES] = df[BINAIRES].astype("int8")
log_step("typage", "Variables nominales en category, binaires en int8", avant, df)

# 5.5 Découpage temporel (colonne ajoutée, jeu conservé complet)
avant = df
df["split"] = np.where(df["month"].isin(TRAIN_MONTHS), "train", "test")
df["split"] = df["split"].astype("category")
log_step("decoupage_temporel", f"Colonne split : train = mois {TRAIN_MONTHS}, test = mois 6 et 7", avant, df)

# Contrôles après transformation
assert df.duplicated().sum() == 0
for c in MANQUANTS:
    assert not (df[c] == -1).any()
print(df.shape)
pd.DataFrame(TRANSFORMATIONS)[["etape", "nom", "lignes_apres", "colonnes_apres"]]

(1000000, 38)


,etape,nom,lignes_apres,colonnes_apres
0,1,doublons,1000000,32
1,2,recodage_manquants,1000000,38
2,3,colonnes_constantes,1000000,37
3,4,typage,1000000,37
4,5,decoupage_temporel,1000000,38


## 6. Export et journal de lineage

In [7]:
clean_path = OUT_DIR / "base_clean.parquet"
df.to_parquet(clean_path, index=False)

lineage = {
    "projet": "Atelier fil rouge M1 - Groupe D - fraude à l'ouverture de compte",
    "execution": {"date_execution": datetime.now(timezone.utc).isoformat(timespec="seconds"),
                  "python": platform.python_version(), "pandas": pd.__version__, "systeme": platform.platform()},
    "sources": [{**SOURCE,
                 "fichiers": [{"fichier": n, "sha256": s} for n, s in parts_sha.items()],
                 "sha256_parties_recollees": rebuilt_sha,
                 "nature": "Données générées à partir d'un jeu réel anonymisé (pas de données réelles)"}],
    "controles_qualite": quality,
    "transformations": TRANSFORMATIONS,
    "sorties": [{"fichier": str(clean_path.relative_to(ROOT)), "sha256": sha256_file(clean_path),
                 "taille_octets": clean_path.stat().st_size}],
}
with open(ROOT / "lineage.json", "w", encoding="utf-8") as f:
    json.dump(lineage, f, indent=2, ensure_ascii=False, default=int)
print("lineage.json écrit ;", clean_path.name, "exporté :", df.shape)

lineage.json écrit ; base_clean.parquet exporté : (1000000, 38)
